# JCSD full pipeline: profile -> quality/acceptance -> joint search -> end-to-end -> calibration -> figures
Runs on Colab or Kaggle (GPU runtime). No Google Drive: artifacts are zipped and downloaded at the end of each
stage, and a later session restores them by uploading that zip (cell 0b).

Run **one platform per session**. Stages 3-4 (quality + acceptance) are hardware independent: run them once and
reuse the zip on the second platform. Stages 2, 5, 6 are per platform.

Rough times on a T4 for pair A (pythia-160m -> pythia-2.8b), CNN/DM: tests 5 min, profile 5 min, quality ~20-30 min
(200 prompts x 10 settings, batched), search 2 min (CPU), e2e ~1-1.5 h (100 prompts x 3 passes x ~10 methods).

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.get_device_name())

## 0. Code
Clone the repo (public). If you have local changes not pushed yet, upload `jcsd.zip` instead and unzip it.

In [ ]:
import os
if not os.path.exists('hdl_project'):
    !git clone -q https://github.com/TakshayBansal/hdl_project
%cd hdl_project/jcsd
!pip -q install accelerate datasets evaluate rouge_score scipy pytest   # keep the preinstalled torch/transformers; versions are logged in every artifact
import transformers; print('transformers', transformers.__version__)

### 0b. (Optional) restore artifacts from an earlier session
Upload `artifacts.zip` produced by the last cell of a previous session.

In [ ]:
try:
    from google.colab import files
    up = files.upload()            # choose artifacts.zip; skip (cancel) if this is the first session
    for f in up:
        !unzip -oq {f}
except ImportError:
    print('Kaggle: add artifacts.zip as a dataset and unzip it here, e.g. !unzip -oq /kaggle/input/<name>/artifacts.zip')
!ls profiles quality search results calib 2>/dev/null | head -50

## 1. Settings

In [ ]:
PAIR = 'A'          # A: pythia-160m -> pythia-2.8b, A410: pythia-410m -> 2.8b, B: Qwen2.5-0.5B-Instruct -> 3B-Instruct
TASK = 'cnndm'      # cnndm (ROUGE-2) or gsm8k (exact match; use with pair B, Pythia scores ~0)
DTYPE = 'float16'
GPU = torch.cuda.get_device_name().replace(' ', '_').lower()
TAG = f'{GPU}_{DTYPE}'      # platform label; a different dtype on the same GPU is a different (declared) platform
CTX = 512 if TASK == 'cnndm' else 1024
COMMON = f'--pair {PAIR} --task {TASK} --dtype {DTYPE}'
print(TAG, COMMON)

## 2. Tests (must pass before anything else)
Slow chi-square tests included; add `-m 'not slow'` for a quick run.

In [ ]:
!python -m pytest -q -x 2>&1 | tail -15

## 3. Tree attention on the real models (spec 8.3)
If it prints FAIL, rerun with `--attn eager`; if that fails too, use pair B.

In [ ]:
!python scripts/check_tree_attention.py {COMMON}

## 4. Hardware profile (per platform)

In [ ]:
!python scripts/run_profile.py {COMMON} --ctx {CTX} --tag {TAG}
!python scripts/run_profile.py {COMMON} --ctx {CTX} --masked --tag {TAG}

## 5. Quality curve + acceptance vectors (hardware independent, run once)
Resumable: finished settings are skipped.

In [ ]:
!python scripts/run_quality.py {COMMON} --n 200 --batch 16

## 6. Joint search (CPU, seconds)

In [ ]:
!python scripts/run_search.py {COMMON} --hw {TAG} --ctx {CTX} --tols 0.02 0.05 0.10

## 7. End-to-end measurement
Add `--transfer search/<other_platform>_..._json` once the other platform's search file is present (H3).

In [ ]:
import glob
other = [f for f in glob.glob(f'search/*_{TASK}.json') if not os.path.basename(f).startswith(TAG)]
print('transfer from', other)
tr = ('--transfer ' + ' '.join(other)) if other else ''
!python scripts/run_e2e.py {COMMON} --hw {TAG} --n 100 --passes 3 {tr}

## 8. Calibrate the cost model, re-search, re-measure (spec 9.2)

In [ ]:
!python scripts/calibrate.py {COMMON} --hw {TAG}
!python scripts/run_search.py {COMMON} --hw {TAG} --ctx {CTX} --tols 0.02 0.05 0.10
!python scripts/run_e2e.py {COMMON} --hw {TAG} --n 100 --passes 3 --tag round2 {tr}

## 9. Figures, hypothesis summary, LaTeX tables

In [ ]:
!python scripts/make_figures.py {COMMON} --out figures
!python scripts/summarize.py {COMMON} --tables tables | tee summary_{TAG}.txt
from IPython.display import Image, display
for f in sorted(glob.glob('figures/*.png')):
    print(f); display(Image(f, width=700))

## 10. Save everything (download; upload again in 0b next session)

In [ ]:
!zip -qr artifacts.zip profiles quality search results calib figures tables summary_*.txt 2>/dev/null
try:
    from google.colab import files; files.download('artifacts.zip')
except ImportError:
    print('Kaggle: download artifacts.zip from the Output panel')